In [1]:
from datetime import date, datetime
from dateutil.relativedelta import relativedelta
from functools import partial
from tenacity import retry, stop_after_delay, wait_fixed, retry_if_exception_type
from eutils import EutilsNCBIError, EutilsRequestError


import csv
import logging
import time
from multiprocessing.pool import ThreadPool
from sys import stdin, stdout
from time import perf_counter
from typing import Iterator, List, Optional

import pandas as pd 
import numpy as np 

from metapub import PubMedFetcher, PubMedArticle
from metapub import pubmedcentral


from sqlitedict import SqliteDict

In [2]:
pmid_clean_list = np.load('PMID_lists/pmids_'+'.npy')

In [4]:

article_cache = SqliteDict("pubmed_cache.db", "articles")
fetcher = PubMedFetcher()


def fetch_article(pmid: str) -> PubMedArticle:
    "call fetcher.article_by_pmid with lots of logging statements"
    # log as this is going to take a non-trivial amount of time
    logging.info("fetching article with pmid=%r", pmid)

    t0 = perf_counter()
    article = fetcher.article_by_pmid(pmid)
    dt = perf_counter() - t0
    logging.info("fetched pmid=%s in %.3fms", pmid, dt * 1000)

    # check if we got the right thing back
    if article.pmid != pmid:
        logging.warning("article with pmid=%r returned pmid=%r", pmid, article.pmid)

    return article


def fetch_article_cached(pmid: str) -> PubMedArticle:
    """cache XML of fetched articles so subsequent runs are faster.
    errors are returned as a new PubMedArticle object with an empty XML string
    """
    try:
        xml = article_cache[pmid]
    except KeyError:
        # cached failed, continue below with fetching
        pass
    else:
        # cache hit, parse it
        return PubMedArticle(xml)

    try:
        article = fetch_article(pmid)
    except Exception as err:
        logging.warning("error fetching pmid=%r: %s", pmid, err)
        article = PubMedArticle("")

    # add a delay to avoid exceeding the API rate limit
    time.sleep(1 / 3)

    # save XML in cache for next time
    article_cache[pmid] = article.xml
    article_cache.commit()

    return article


def fetch_articles(
    pmids: List[str], *, processes: Optional[int] = None
) -> Iterator[PubMedArticle]:
    "use a threadpool to fetch articles from metapub in parallel, caching where possible"
    with ThreadPool(processes=processes) as pool:
        for article in pool.imap_unordered(fetch_article_cached, pmids):
            # couldn't be fetched
            if article is None:
                continue
            yield article


def main() -> None:
    "fetch articles by PMID from a list called pmid_clean_list and write to a CSV file"
    with open("/path/to/output.csv", "w", newline="", encoding="utf-8") as f:
        out = csv.writer(f)
        first = True
        for article in fetch_articles(pmid_clean_list, processes=6):
            row = dict(
                pmid=article.pmid,
                pmc=article.pmc,
                doi=article.doi,
                date=article.history,
                title=article.title,
                authors=article.authors,
                journal=article.journal,
                abstract=article.abstract
            )
            if first:
                out.writerow(row.keys())
                first = False
            out.writerow(row.values())


if __name__ == "__main__":
    logging.basicConfig(level=logging.DEBUG)
    main()

2024-04-16 00:13:51 LAPTOP-S8N3C7A8 root[15700] INFO fetching article with pmid='30926424'
2024-04-16 00:13:51 LAPTOP-S8N3C7A8 root[15700] INFO fetching article with pmid='31866010'
2024-04-16 00:13:51 LAPTOP-S8N3C7A8 root[15700] INFO fetching article with pmid='24942373'
2024-04-16 00:13:51 LAPTOP-S8N3C7A8 root[15700] INFO fetching article with pmid='36924766'
2024-04-16 00:13:51 LAPTOP-S8N3C7A8 root[15700] INFO fetching article with pmid='26764136'
2024-04-16 00:13:51 LAPTOP-S8N3C7A8 root[15700] INFO fetching article with pmid='12032081'
2024-04-16 00:13:52 LAPTOP-S8N3C7A8 root[15700] INFO fetched pmid=36924766 in 667.421ms
2024-04-16 00:13:52 LAPTOP-S8N3C7A8 root[15700] INFO fetched pmid=26764136 in 727.460ms
2024-04-16 00:13:52 LAPTOP-S8N3C7A8 root[15700] INFO fetched pmid=12032081 in 742.949ms
2024-04-16 00:13:52 LAPTOP-S8N3C7A8 root[15700] INFO fetched pmid=31866010 in 779.727ms
2024-04-16 00:13:52 LAPTOP-S8N3C7A8 root[15700] INFO fetched pmid=24942373 in 772.392ms
2024-04-16 00:

MetaPubError: Cannot build MetaPubObject; xml string was empty

2024-04-16 00:17:45 LAPTOP-S8N3C7A8 root[15700] INFO fetched pmid=28167801 in 721.860ms


2024-04-16 00:17:45 LAPTOP-S8N3C7A8 root[15700] INFO fetched pmid=18288523 in 784.731ms
